# LangGraph migration spike

This notebook explores wrapping the existing aptitude-search pipeline in a **linear LangGraph**.

Today `run_pipeline` is already a straight chain:

```text
prepare_resume → stage1 → stage2 → stage3 → result
```

LangGraph’s job here is **orchestration + shared state**, not rewriting discovery, search, or validation. Stage internals stay plain Python; graph nodes are thin wrappers that read/write shared state and call the existing stage functions.

## Goal for this notebook

1. Define a `PipelineState` that mirrors what the API already returns (plus inputs).
2. Build a small linear graph (`START → … → END`) that wraps the current stage functions.
3. Confirm `graph.invoke({...})` returns the same shape as today’s `PipelineResult`.

Out of scope for v1: LangChain chat models, ReAct/tool-calling agents, checkpointers, human-in-the-loop, parallel branches, or subgraphs.

See `backend/docs/langgraph-migration.md` for the full plan.

## Setup

Academy-style env helpers (see langchain-academy notebooks). Prompt for `OPENAI_API_KEY` via `getpass` if unset. Pipeline LLM keys still live in `backend/config.toml`; LangSmith tracing below is optional while we explore the graph.

In [ ]:
import getpass
import os

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("OPENAI_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "aptitude-search-langgraph"